# SOLSTICE sources GNN: plasma state -> EIRENE source terms (DIII-D)

The EIRENE-replacement task: predict the neutral volumetric sources
(sp, sne, qe, qi, sm) from the local plasma state on the mesh graph.
Successor of the solpex-paper EIRENE GNN, on the canonical store.

Node features: geometry (psi_n, |B|) + local plasma (te, ti, ne, na_D0,
na_D1, ua_D1). Control parameters additionally condition via FiLM
(set `USE_PARAMS = False` for a purely local model, e.g. for inline
B2.5 coupling studies). Requires `solstice_store_diiid_appfpp_v1.nc`
(the sources-enabled store) in Drive.


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('colab')
!pip -q install git+https://{token}@github.com/abdoudiaw/solstice.git torch_geometric xarray netcdf4

STORE = '/content/drive/MyDrive/SOLPS_DATA/solstice_store_diiid_appfpp_v1.nc'
WEIGHTS_DEST = '/content/drive/MyDrive/SOLPS_DATA/solstice_weights/gnn_sources'

MODEL      = 'gnn'         # released architecture (legacy: 'gnn_v1' native-mesh)
USE_PARAMS = True
HIDDEN     = 128
LAYERS     = 6
N_LATENT   = 256
EPOCHS     = 300
BATCH      = 16
LR         = 1e-3


In [ ]:
import numpy as np, xarray as xr, torch, torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from solstice.graphs import cell_adjacency_edges, default_node_features, build_latent_graph
from solstice.models import build_model

ds = xr.open_dataset(STORE)
N_CELLS = ds.sizes['cell']
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(dict(ds.sizes), '| device:', device)


## Features and targets

Plasma node features: log10 for decade-spanning positive fields, then
global standardization. Source targets: per-cell standardization
(handles the orders-of-magnitude spatial structure; signed values
like qe stay linear).


In [ ]:
PLASMA = {'te': True, 'ti': True, 'ne': True, 'na_D0': True, 'na_D1': True, 'ua_D1': False}
SOURCES = ['sp', 'sne', 'qe', 'qi', 'sm']

geom = default_node_features(ds)
geom_mean, geom_std = geom.mean(0), geom.std(0) + 1e-12
geom = (geom - geom_mean) / geom_std

XF = [np.tile(geom.T[:, None, :], (1, ds.sizes['case'], 1))[i] for i in range(2)]
pf_mean, pf_std = {}, {}
for name, log in PLASMA.items():
    a = ds[name].values.astype(np.float64)
    if log:
        a = np.log10(np.clip(np.abs(a), 1e-6, None))
    pf_mean[name], pf_std[name] = float(a.mean()), float(a.std() + 1e-12)
    XF.append((a - pf_mean[name]) / pf_std[name])
XF = np.stack(XF, axis=2).astype(np.float32)   # (case, cell, F) after transpose fix
XF = XF.transpose(1, 0, 2) if XF.shape[0] == 2 else XF
print('node features:', XF.shape)

Y, y_mean, y_std = [], [], []
for name in SOURCES:
    y = ds[name].values.astype(np.float64)
    m, s = y.mean(0), y.std(0) + 1e-12
    Y.append((y - m) / s); y_mean.append(m); y_std.append(s)
Y = np.stack(Y, axis=2)
y_mean, y_std = np.stack(y_mean, 1), np.stack(y_std, 1)

raw = {v[6:]: ds[v].values.astype(np.float64) for v in ds.data_vars if v.startswith('input_')}
for name in list(raw):
    if np.unique(raw[name]).size == 1: del raw[name]
if 'pe' in raw and 'pi' in raw and np.array_equal(raw['pe'], raw['pi']):
    raw['ptot'] = raw.pop('pe') + raw.pop('pi')
if 'hci' in raw and 'hce' in raw and np.array_equal(raw['hci'], raw['hce']):
    raw['chi'] = raw.pop('hci'); del raw['hce']
INPUTS = sorted(raw)
X = np.stack([raw[v] for v in INPUTS], axis=1)
for j, v in enumerate(INPUTS):
    if v in ('core_fueling', 'puff_D2', 'puff_Ne'):
        X[:, j] = np.log10(np.clip(X[:, j], 1e-30, None))
x_mean, x_std = X.mean(0), X.std(0) + 1e-12
x_min, x_max = X.min(0), X.max(0)   # training box, for out-of-range warnings
Xn = ((X - x_mean) / x_std) if USE_PARAMS else np.zeros_like(X)

rng = np.random.default_rng(0)
idx = rng.permutation(ds.sizes['case'])
split = int(0.85 * len(idx))
itr, ite = idx[:split], idx[split:]
print(len(itr), 'train /', len(ite), 'test |', len(INPUTS), 'params, FiLM', USE_PARAMS)


## Graph and batching (node features vary per case)


In [ ]:
ei_np, ea_np = cell_adjacency_edges(ds)
ea_np = ea_np / np.abs(ea_np).max(axis=0)
if MODEL != 'gnn_v1':
    lg = build_latent_graph(ds.cell_r.values, ds.cell_z.values, n_latent=N_LATENT, k_nn=6)
    la_np = lg['latent_attr'] / np.abs(lg['latent_attr']).max(axis=0)
    aa_np = lg['assign_attr'] / (np.abs(lg['assign_attr']).max(axis=0) + 1e-12)

def batch_graph(case_ids):
    B = len(case_ids)
    xb = torch.tensor(XF[case_ids].reshape(B * N_CELLS, -1), device=device)
    pb = torch.tensor(Xn[case_ids], dtype=torch.float32, device=device)
    yb = torch.tensor(Y[case_ids].reshape(B * N_CELLS, -1), dtype=torch.float32, device=device)
    if MODEL == 'gnn_v1':
        ei = torch.tensor(np.concatenate([ei_np + b * N_CELLS for b in range(B)], axis=1), device=device)
        ea = torch.tensor(np.tile(ea_np, (B, 1)), dtype=torch.float32, device=device)
        params = pb.repeat_interleave(N_CELLS, dim=0)
        return dict(x=xb, edge_index=ei, edge_attr=ea, params=params), yb
    ai = torch.tensor(np.concatenate(
        [lg['assign_index'] + np.array([[b * N_CELLS], [b * N_LATENT]]) for b in range(B)], axis=1), device=device)
    aa = torch.tensor(np.tile(aa_np, (B, 1)), dtype=torch.float32, device=device)
    le = torch.tensor(np.concatenate([lg['latent_edges'] + b * N_LATENT for b in range(B)], axis=1), device=device)
    la = torch.tensor(np.tile(la_np, (B, 1)), dtype=torch.float32, device=device)
    pl = pb.repeat_interleave(N_LATENT, dim=0)
    return dict(x=xb, assign_index=ai, assign_attr=aa, latent_edges=le,
                latent_attr=la, params_latent=pl, n_latent=B * N_LATENT), yb


## Train


In [ ]:
NF = XF.shape[2]
cfg = {'node_features': NF, 'param_dim': len(INPUTS), 'out_features': len(SOURCES), 'hidden': HIDDEN}
cfg |= {'n_layers': LAYERS} if MODEL == 'gnn_v1' else {'n_process_layers': LAYERS}
model = build_model(MODEL, cfg).to(device)
print(MODEL, sum(p.numel() for p in model.parameters()), 'parameters')

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
kw_val, y_val = batch_graph(ite)
best, best_state = np.inf, None
for ep in range(EPOCHS):
    model.train()
    perm = rng.permutation(itr)
    tr = 0.0
    for i in range(0, len(perm), BATCH):
        kw, yb = batch_graph(perm[i:i + BATCH])
        opt.zero_grad()
        loss = nn.functional.mse_loss(model(**kw), yb)
        loss.backward(); opt.step()
        tr += loss.item() * len(perm[i:i + BATCH])
    sched.step()
    model.eval()
    with torch.no_grad():
        val = nn.functional.mse_loss(model(**kw_val), y_val).item()
    if val < best:
        best, best_state = val, {k: v.clone() for k, v in model.state_dict().items()}
    if ep % 10 == 0:
        print(f'ep{ep}: train {tr/len(perm):.4f} val {val:.4f}')
model.load_state_dict(best_state)
print('best val MSE (normalized):', round(best, 5))


## Metrics (standardized units) and plots


In [ ]:
from skimage.metrics import structural_similarity as ssim
import json as _json
FACE_SETS = _json.loads(ds.attrs['face_sets'])
NX, NY = ds.attrs['nx'], ds.attrs['ny']
def to_image(v):
    img = np.full((NX, NY), np.nan)
    img[ds.cell_ix.values - 1, ds.cell_iy.values - 1] = v
    return img
def target_cells(which):
    f = ds.face_set.values == FACE_SETS[which]
    c = ds.face_cells.values[f, 0]
    return c[np.argsort(ds.cell_iy.values[c])]
INNER, OUTER = target_cells('inner_target'), target_cells('outer_target')

model.eval()
with torch.no_grad():
    kw, _ = batch_graph(ite)
    P_all = model(**kw).cpu().numpy().reshape(len(ite), N_CELLS, len(SOURCES))
T_all = Y[ite]
print(f"{'source':7s} {'SSIM':>6s} {'R2':>7s} {'RMSE(sig)':>10s} {'peak-tgt %':>11s}")
for j, name in enumerate(SOURCES):
    P, T = P_all[:, :, j], T_all[:, :, j]
    ss = np.mean([ssim(to_image(t), to_image(p), data_range=np.nanmax(t)-np.nanmin(t))
                  for t, p in zip(T, P)])
    r2 = 1 - np.sum((P - T)**2) / np.sum((T - T.mean())**2)
    rmse = np.sqrt(np.mean((P - T)**2))
    Pp = P * y_std[:, j] + y_mean[:, j]; Tp = T * y_std[:, j] + y_mean[:, j]
    peak = np.median([abs(np.abs(pp[c]).max() - np.abs(tt[c]).max()) / (np.abs(tt[c]).max() + 1e-30) * 100
                      for tt, pp in zip(Tp, Pp) for c in (INNER, OUTER)])
    print(f'{name:7s} {ss:6.3f} {r2:7.3f} {rmse:10.3f} {peak:11.2f}')


In [ ]:
def plot_field(values, title='', ax=None, cmap='viridis', sym=False):
    verts = np.stack([ds.cell_corners_r.values, ds.cell_corners_z.values], axis=-1)
    ax = ax or plt.subplots(figsize=(4, 6))[1]
    kwp = dict(clim=(-np.max(np.abs(values)), np.max(np.abs(values)))) if sym else {}
    pc = PolyCollection(verts, array=values, cmap=cmap, edgecolor='none', **kwp)
    ax.add_collection(pc); ax.autoscale(); ax.set_aspect('equal')
    ax.set_title(title); plt.colorbar(pc, ax=ax, shrink=0.8)

kk, j = 0, SOURCES.index('sp')
tru = T_all[kk, :, j] * y_std[:, j] + y_mean[:, j]
prd = P_all[kk, :, j] * y_std[:, j] + y_mean[:, j]
fig, axs = plt.subplots(1, 3, figsize=(13, 6))
plot_field(np.log10(np.clip(np.abs(tru), 1e15, None)), 'log10|sp| SOLPS', axs[0])
plot_field(np.log10(np.clip(np.abs(prd), 1e15, None)), f'log10|sp| {MODEL}', axs[1])
plot_field(prd - tru, 'error (1/m^3s)', axs[2], cmap='RdBu_r', sym=True)
plt.tight_layout()


## Save


In [ ]:
import os
os.makedirs(WEIGHTS_DEST, exist_ok=True)
torch.save({'state_dict': model.state_dict(), 'model_class': MODEL, 'config': cfg,
            'task': 'sources', 'plasma_features': PLASMA, 'sources': SOURCES,
            'use_params': USE_PARAMS, 'inputs': INPUTS,
            'x_mean': x_mean, 'x_std': x_std, 'x_min': x_min, 'x_max': x_max,
            'geom_mean': geom_mean, 'geom_std': geom_std,
            'pf_mean': pf_mean, 'pf_std': pf_std,
            'y_mean': y_mean, 'y_std': y_std,
            'n_latent': N_LATENT if MODEL != 'gnn_v1' else None,
            'epochs': EPOCHS, 'seed': 0},
           f'{WEIGHTS_DEST}/pepc-diiid-sources.pt')
print(os.listdir(WEIGHTS_DEST))
